###  Build Full Dataset from Multiple Patients

Now we'll process multiple patients and handle the **class imbalance problem**:
- Seizures are rare (~1%)
- We'll balance by taking equal seizure/normal samples
- This prevents the model from just predicting "normal" always

In [4]:
%load_ext autoreload
%autoreload 2

import gc
import pandas as pd
import handy.my_utils as deniz
import os
import warnings
import mne
from EEGPreprocessor import EEGPreprocessor
import matplotlib.pyplot as plt
from scipy.signal import welch
import numpy as np

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# df = pd.read_csv('Epileptic Seizure Recognition.csv')
CHB_MIT_PATH = 'data-understanding/data/chb-mit'
preprocessor = EEGPreprocessor(sampling_rate=256, target_sfreq=128)
new_fs = preprocessor.target_sfreq
all_annotations = {}
patient_dirs = sorted([d for d in os.listdir(CHB_MIT_PATH)
                       if os.path.isdir(os.path.join(CHB_MIT_PATH, d))
                       and d.startswith('chb')])

total_seizures = 0
total_files_with_seizures = 0

print(f"\n👥 Processing {len(patient_dirs)} patients...\n")

for patient in patient_dirs:
    patient_path = os.path.join(CHB_MIT_PATH, patient)
    summary_file = os.path.join(patient_path, f'{patient}-summary.txt')

    if os.path.exists(summary_file):
        seizure_info = deniz.parse_summary_file(summary_file)

        if seizure_info:
            all_annotations[patient] = seizure_info
            n_files = len(seizure_info)
            n_seizures = sum(len(times) for times in seizure_info.values())
            total_files_with_seizures += n_files
            total_seizures += n_seizures

            print(f"   ✓ {patient}: {n_files} files, {n_seizures} seizures")
        else:
            print(f"   - {patient}: No seizures found")
    else:
        print(f"   ✗ {patient}: No summary file")

   ✓ chb01: 7 files, 7 seizures
   ✓ chb02: 3 files, 3 seizures
   ✓ chb03: 7 files, 7 seizures
   ✓ chb04: 3 files, 4 seizures
   ✓ chb05: 5 files, 5 seizures
   ✓ chb06: 7 files, 10 seizures
   ✓ chb07: 3 files, 3 seizures
   ✓ chb08: 5 files, 5 seizures
   ✓ chb09: 3 files, 4 seizures
   ✓ chb10: 7 files, 7 seizures
   ✓ chb11: 3 files, 3 seizures
   ✓ chb12: 13 files, 40 seizures
   ✓ chb13: 8 files, 12 seizures
   ✓ chb14: 7 files, 8 seizures
   ✓ chb15: 14 files, 20 seizures
   ✓ chb16: 6 files, 10 seizures
   ✓ chb17: 3 files, 3 seizures
   ✓ chb18: 6 files, 6 seizures
   ✓ chb19: 3 files, 3 seizures
   ✓ chb20: 6 files, 8 seizures
   ✓ chb21: 4 files, 4 seizures
   ✓ chb22: 3 files, 3 seizures
   ✓ chb23: 3 files, 7 seizures
   ✓ chb24: 12 files, 16 seizures


In [6]:
edf_file = all_annotations['chb16'].keys()
for edf in edf_file:
    print(edf)

chb16_10.edf
chb16_11.edf
chb16_14.edf
chb16_16.edf
chb16_17.edf
chb16_18.edf


In [7]:
print(all_annotations)

In [8]:
exclude_list = [
    'chb12/chb12_29.edf',
    'chb12/chb12_27.edf',
    'chb12/chb12_28.edf'
]

# Sözlüğü güvenli bir şekilde güncellemek için iç içe döngü kullanıyoruz
for item in exclude_list:
    # item örneği: 'chb12/chb12_29.edf'
    # folder: 'chb12', filename: 'chb12_29.edf'
    folder, filename = item.split('/')

    # Eğer bu klasör ana sözlükte varsa ve dosya o klasörün içindeyse sil
    if folder in all_annotations and filename in all_annotations[folder]:
        del all_annotations[folder][filename]

# Sonucu doğrulamak için chb12 klasörüne bakalım
print(all_annotations['chb12'].keys())

dict_keys(['chb12_06.edf', 'chb12_08.edf', 'chb12_09.edf', 'chb12_10.edf', 'chb12_11.edf', 'chb12_23.edf', 'chb12_33.edf', 'chb12_36.edf', 'chb12_38.edf', 'chb12_42.edf'])


In [9]:
PATIENTS_TO_USE = ['chb01', 'chb03', 'chb06', 'chb10', 'chb14', 'chb15', 'chb16', 'chb24']
for i in PATIENTS_TO_USE:
    edf_file = all_annotations[i].keys()
    for edf in edf_file:
        print(edf)

chb01_03.edf
chb01_04.edf
chb01_15.edf
chb01_16.edf
chb01_18.edf
chb01_21.edf
chb01_26.edf
chb03_01.edf
chb03_02.edf
chb03_03.edf
chb03_04.edf
chb03_34.edf
chb03_35.edf
chb03_36.edf
chb06_01.edf
chb06_04.edf
chb06_09.edf
chb06_10.edf
chb06_13.edf
chb06_18.edf
chb06_24.edf
chb10_12.edf
chb10_20.edf
chb10_27.edf
chb10_30.edf
chb10_31.edf
chb10_38.edf
chb10_89.edf
chb14_03.edf
chb14_04.edf
chb14_06.edf
chb14_11.edf
chb14_17.edf
chb14_18.edf
chb14_27.edf
chb15_06.edf
chb15_10.edf
chb15_15.edf
chb15_17.edf
chb15_20.edf
chb15_22.edf
chb15_28.edf
chb15_31.edf
chb15_40.edf
chb15_46.edf
chb15_49.edf
chb15_52.edf
chb15_54.edf
chb15_62.edf
chb16_10.edf
chb16_11.edf
chb16_14.edf
chb16_16.edf
chb16_17.edf
chb16_18.edf
chb24_01.edf
chb24_03.edf
chb24_04.edf
chb24_06.edf
chb24_07.edf
chb24_09.edf
chb24_11.edf
chb24_13.edf
chb24_14.edf
chb24_15.edf
chb24_17.edf
chb24_21.edf


In [10]:
# inspecting channel names

PATIENTS_TO_USE = ['chb03']
CHB_MIT_PATH = 'data-understanding/data/chb-mit'

for patient_id in PATIENTS_TO_USE:
    # Sözlük anahtarlarını doğrudan döngüye sokabiliriz
    file_names = all_annotations[patient_id].keys()

    for file_name in file_names:
        # Yol birleştirme hatası düzeltildi
        sample_path = os.path.join(CHB_MIT_PATH, patient_id, file_name)

        try:
            signals, channel_names, fs = deniz.load_edf_file(sample_path)
            print(f"Hasta: {patient_id} | Dosya: {file_name}")
            print(f"Kanallar: {channel_names}")
            print("Kanal sayısı: ", len(channel_names))
            print('-' * 20)

            # Belleği temiz tutmak için objeleri silelim
            del signals, channel_names, fs
        except Exception as e:
            print(f"Hata oluştu ({file_name}): {e}")


Hasta: chb03 | Dosya: chb03_36.edf
Kanallar: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8']
Kanal sayısı:  23
--------------------


In [11]:
deniz.load_edf_file('data-understanding/data/chb-mit/chb01/chb01_03.edf')

(array([[-17.77777778,   0.1953602 ,   0.1953602 , ..., -17.77777778,
          -8.79120879,  -4.88400488],
        [ 39.26739927,   0.1953602 ,   0.1953602 , ..., -23.24786325,
         -22.46642247, -20.51282051],
        [ -3.71184371,   0.1953602 ,   0.1953602 , ...,  59.19413919,
          60.36630037,  59.19413919],
        ...,
        [ -9.18192918,   0.1953602 ,   0.1953602 , ...,  48.64468864,
          27.93650794,  12.6984127 ],
        [-39.65811966,   0.1953602 ,   0.1953602 , ...,  43.56532357,
          38.48595849,  36.14163614],
        [-59.97557998,   0.1953602 ,   0.1953602 , ...,  19.34065934,
          26.37362637,  32.23443223]]),
 ['FP1-F7',
  'F7-T7',
  'T7-P7',
  'P7-O1',
  'FP1-F3',
  'F3-C3',
  'C3-P3',
  'P3-O1',
  'FP2-F4',
  'F4-C4',
  'C4-P4',
  'P4-O2',
  'FP2-F8',
  'F8-T8',
  'T8-P8',
  'P8-O2',
  'FZ-CZ',
  'CZ-PZ',
  'P7-T7',
  'T7-FT9',
  'FT9-FT10',
  'FT10-T8',
  'T8-P8'],
 256)

In [12]:
FINAL_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]
signals, channel_names, fs = deniz.fix_eeg_channels_load_edf(
    file_path='data-understanding/data/chb-mit/chb01/chb01_03.edf', final_channels=FINAL_CHANNELS,
    load_func=deniz.load_edf_file)

In [13]:
signals.shape

(18, 921600)

In [14]:
channel_names

['FP1-F7',
 'F7-T7',
 'T7-P7',
 'P7-O1',
 'FP1-F3',
 'F3-C3',
 'C3-P3',
 'P3-O1',
 'FP2-F4',
 'F4-C4',
 'C4-P4',
 'P4-O2',
 'FP2-F8',
 'F8-T8',
 'T8-P8',
 'P8-O2',
 'FZ-CZ',
 'CZ-PZ']

In [15]:
PATIENTS_TO_USE = ['chb01', 'chb03', 'chb06', 'chb10', 'chb14', 'chb15', 'chb16', 'chb24']
CHB_MIT_PATH = 'data-understanding/data/chb-mit'
FINAL_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

for patient_id in PATIENTS_TO_USE:
    file_names = all_annotations[
        patient_id].keys()  # bu şekilde hastanın icindeki edf dosyalarını çekebiliyoruz mesela file_names ilk elemanı chb_01_03.edf

signals, channel_names, fs = deniz.load_edf_file('data-understanding/data/chb-mit/chb01/chb01_03.edf')

In [16]:
signals.shape

(23, 921600)

In [18]:
N_CHANNELS = 18
print(channel_names)
print('*' * 60)
print(channel_names[:N_CHANNELS:])

['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8']
************************************************************
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']


In [19]:


# Storage
all_seizure_windows = []
all_normal_windows = []

WINDOW_SIZE = 2  # seconds
OVERLAP = 0.5  # 50% overlap
WINDOW_SAMPLES = WINDOW_SIZE * new_fs  # 2 * 128 = 256 samples
STEP_SAMPLES = int(WINDOW_SAMPLES * (1 - OVERLAP))

# Configuration
PATIENTS_TO_USE = ['chb01', 'chb03', 'chb06', 'chb10', 'chb14', 'chb15', 'chb16', 'chb24']

N_CHANNELS = 18

# Storage
all_seizure_windows = []
all_normal_windows = []

print(f"\n👥 Processing {len(PATIENTS_TO_USE)} patients...")
print("-" * 50)

normal_counter = 0  # Counter for sampling normal windows

for patient in PATIENTS_TO_USE:
    if patient not in all_annotations:
        continue

    patient_seizure = 0
    patient_normal = 0

    for filename, seizure_times in all_annotations[patient].items():
        file_path = os.path.join(CHB_MIT_PATH, patient, filename)

        if not os.path.exists(file_path):
            continue

        try:
            # Load file
            signals_temp, ch_names, fs_temp = deniz.load_edf_file(file_path)
            if signals_temp is None:
                continue

            # Use only N_CHANNELS
            signals_temp = signals_temp[:N_CHANNELS, :]

            # Preprocess
            signals_temp = preprocessor.preprocess(signals_temp)

            n_samples = signals_temp.shape[1]

            # Extract windows
            for start in range(0, n_samples - WINDOW_SAMPLES, STEP_SAMPLES):
                end = start + WINDOW_SAMPLES
                window = signals_temp[:, start:end]

                if window.shape[1] != WINDOW_SAMPLES:
                    continue

                # Check label
                window_start_sec = start / fs
                window_end_sec = end / fs

                is_seizure = False
                for sz_start, sz_end in seizure_times:
                    if window_start_sec < sz_end and window_end_sec > sz_start:
                        overlap_start = max(window_start_sec, sz_start)
                        overlap_end = min(window_end_sec, sz_end)
                        if (overlap_end - overlap_start) / WINDOW_SIZE >= 0.5:
                            is_seizure = True
                            break

                if is_seizure:
                    all_seizure_windows.append(window.astype(np.float32))
                    patient_seizure += 1
                else:
                    # Keep every 20th normal window
                    normal_counter += 1
                    if normal_counter % 20 == 0:
                        all_normal_windows.append(window.astype(np.float32))
                        patient_normal += 1

            # Clear memory
            del signals_temp
            gc.collect()

        except Exception as e:
            continue

    print(f"   ✓ {patient}: {patient_seizure} seizure, {patient_normal} normal")
    gc.collect()

print("\n" + "-" * 50)

n_seizure = len(all_seizure_windows)
n_normal_available = len(all_normal_windows)

print(f"📊 Collected: {n_seizure} seizure, {n_normal_available} normal")

# Balance: use all seizures + 2x normals (or whatever is available)
n_normal_to_use = min(n_normal_available, n_seizure * 2)

# Sample normal windows
np.random.seed(42)
if n_normal_available > n_normal_to_use:
    normal_indices = np.random.choice(n_normal_available, n_normal_to_use, replace=False)
    X_normal = np.array([all_normal_windows[i] for i in normal_indices], dtype=np.float32)
else:
    X_normal = np.array(all_normal_windows, dtype=np.float32)

X_seizure = np.array(all_seizure_windows, dtype=np.float32)

# Clear lists
del all_seizure_windows, all_normal_windows
gc.collect()

# Combine
X = np.concatenate([X_seizure, X_normal], axis=0)
y = np.concatenate([np.ones(len(X_seizure)), np.zeros(len(X_normal))], axis=0)

del X_seizure, X_normal
gc.collect()

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

print(f"\n📈 Final Dataset:")
print(f"   • Total: {len(X)} samples")
print(f"   • Seizure: {int(np.sum(y))} ({100 * np.sum(y) / len(y):.1f}%)")
print(f"   • Normal: {int(len(y) - np.sum(y))} ({100 * (len(y) - np.sum(y)) / len(y):.1f}%)")
print(f"   • Shape: {X.shape}")
print(f"   • Memory: ~{X.nbytes / 1024 / 1024:.1f} MB")

print("\n" + "=" * 60)
print("✅ Dataset ready!")
print("=" * 60)


📈 Final Dataset:
   • Total: 15129 samples
   • Seizure: 5043 (33.3%)
   • Normal: 10086 (66.7%)
   • Shape: (15129, 18, 256)
   • Memory: ~265.9 MB

✅ Dataset ready!
